In [ ]:
import json, fnmatch

d = json.load(open('F:/supremeai backup/backend/coverage.json'))
files = d['files']
PKGS = ("core", "api", "tools", "ws", "workers")
OMIT = ["*/tests/*","*/venv/*","*/env/*",".venv/*","*/__pycache__/*","*/migrations/*","*/alembic/*","*/__init__.py","tools/cli.py","core/worker_main.py"]
om = lambda p: any(fnmatch.fnmatch(p, o) for o in OMIT)

rows = []
tc = ts = 0
for path, info in files.items():
    if not path.startswith(PKGS):
        continue
    if om(path):
        continue
    pct = info.get('summary', {}).get('percent_covered')
    if pct is None:
        pct = info.get('percent_covered', 0.0)
    ns = info.get('summary', {}).get('num_statements', 0)
    missing = info.get('summary', {}).get('missing_lines', 0)
    rows.append((path, pct, ns, missing))
    tc += (ns - missing)
    ts += ns

rows.sort(key=lambda r: r[1])
print("=== BOTTOM 50 FILES (ascending percent_covered) ===")
print(f"{'FILE':60} {'%COV':>7} {'STMT':>6} {'MISS':>6}")
for path, pct, ns, missing in rows[:50]:
    print(f"{path[:60]:60} {pct:7.2f} {ns:6d} {missing:6d}")
print()
print("TOTAL_INCLUDED_FILES", len(rows))
print("OVERALL_COMBINED_PCT", round(100.0*tc/ts, 2), f"({tc}/{ts})")

from collections import defaultdict
pkg_cov = defaultdict(lambda: [0,0])
for path, pct, ns, missing in rows:
    pkg = path.split('\\')[0] if '\\' in path else path.split('/')[0]
    pkg_cov[pkg][0] += (ns - missing)
    pkg_cov[pkg][1] += ns
print()
print("=== PACKAGE SUMMARY (ascending) ===")
for pkg in sorted(pkg_cov, key=lambda k: 100.0*pkg_cov[k][0]/pkg_cov[k][1]):
    c, s = pkg_cov[pkg]
    print(f"{pkg:10} {100.0*c/s:7.2f}%  files={sum(1 for r in rows if (r[0].split(chr(92))[0] if chr(92) in r[0] else r[0].split('/')[0])==pkg)}  cov={c}/{s}")